In [ ]:
# Montowanie Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import shutil
from pathlib import Path

# Repo i galaz
REPO_URL = "https://github.com/Slasq/Battleships.git"
BRANCH = "RL-and-DQN"

# Kod na dysku Colaba
PROJECT = Path("/content/Battleships")

# Wyniki na Drive
# Stad Drive Desktop zaciaga je na PC
DRIVE_ROOT = Path("/content/drive/MyDrive/Battleships")
DRIVE_MODELS = DRIVE_ROOT / "models"
DRIVE_PLOTS = DRIVE_ROOT / "plots"

# Foldery na wyniki
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
DRIVE_PLOTS.mkdir(parents=True, exist_ok=True)

# Klon raz na sesje, potem sam pull
os.chdir("/content")
if PROJECT.is_dir() and (PROJECT / "ml").is_dir():
    print(f"Projekt już jest: {PROJECT}, ciągnę zmiany")
    os.chdir(PROJECT)
    !git pull --ff-only
    os.chdir("/content")
else:
    !git clone --branch {BRANCH} --single-branch {REPO_URL} Battleships

# Czy projekt jest na miejscu
assert (PROJECT / "ml" / "dqn" / "train.py").exists(), (
    "Brak projektu. Sprawdź clone albo Upload to Colab."
)

os.chdir(PROJECT)
print("CWD:", os.getcwd())

# Trening na CPU nie ma sensu, więc karta sprawdzana od razu
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: BRAK. Środowisko wykonawcze > Zmień typ środowiska > GPU")
print("Wyniki → Drive:", DRIVE_ROOT)
print("Na PC szukaj w folderze Google Drive: My Drive/Battleships/")

In [ ]:
def sync_artifacts_to_drive():
    # Kopiowanie models i plots na Drive
    local_models = PROJECT / "models"
    local_plots = PROJECT / "plots"

    # Modele
    if local_models.exists():
        shutil.copytree(local_models, DRIVE_MODELS, dirs_exist_ok=True)
        print("Zsynchronizowano models →", DRIVE_MODELS)
    else:
        print("Brak folderu models — pomijam")

    # Wykresy
    if local_plots.exists():
        shutil.copytree(local_plots, DRIVE_PLOTS, dirs_exist_ok=True)
        print("Zsynchronizowano plots →", DRIVE_PLOTS)
    else:
        print("Brak folderu plots — pomijam")

    print("Gotowe. Drive Desktop zaraz zaciągnie pliki na komputer.")

print("sync_artifacts_to_drive() gotowe do użycia po treningu")

In [ ]:
# Trening DQN
os.chdir(PROJECT)
!python ml/dqn/train.py
sync_artifacts_to_drive()

In [ ]:
# Trening ProbMap
os.chdir(PROJECT)
!python ml/probmap/train.py
sync_artifacts_to_drive()

In [ ]:
# Ewaluacja
os.chdir(PROJECT)
!python ml/evaluate.py
sync_artifacts_to_drive()